# Lab | Embeddings
## Find the answer before the customer finishes typing

You work for a digital bank. Every day, thousands of customers write to support: *my card has not arrived*, *I was charged twice*, *why is the exchange rate different?* Most of these questions have been asked, and answered, many times before.

The support team wants a tool that takes a new message and instantly shows the agent the **most similar past messages** and their category, so the agent can reuse the answer. Later, the same search will feed a chatbot (that is Day 5's RAG). You will build that tool with **sentence embeddings** and check, with numbers, whether it beats a classic TF-IDF search.

**The data** is the Banking77 dataset: 13,083 real customer messages to a bank, each labelled with one of 77 categories (called **intents**), such as `card_arrival` or `exchange_rate`. `data/banking_train.csv` has 10,003 messages, which play the role of the past messages, and `data/banking_test.csv` has 3,080 messages, which play the role of new ones.

**This lab practises** semantic search with sentence embeddings. You will use, from today's **Embeddings and Semantic Search** lesson: `SentenceTransformer.encode`, cosine similarity, unit-length vectors and the dot product, top-*k* search with `np.argsort`, the TF-IDF search, and KMeans with the adjusted Rand index.

**By the end, you can:**

1. Implement a top-*k* semantic search with sentence embeddings and cosine similarity.
2. Evaluate it against a TF-IDF baseline and explain where each method fails.

Each challenge ends with a **checkpoint** that tells you what you should see. If your numbers are very different, fix that before moving on.

## Setup

The first run downloads the `all-MiniLM-L6-v2` model (about 90 MB, internet needed once). If the import fails, install the library from a terminal with `pip install sentence-transformers`.

In [ ]:
import os
import logging
os.environ["HF_HUB_VERBOSITY"] = "error"      # hide Hugging Face download notices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

pd.set_option("display.max_colwidth", 90)

In [ ]:
train = pd.read_csv("../data/banking_train.csv")
test = pd.read_csv("../data/banking_test.csv")
print(train.shape, test.shape, train["category"].nunique())
train.head()

**Checkpoint:** `(10003, 2) (3080, 2) 77`, and two columns, `text` and `category`.

## Explore the data

1. How many training messages does each intent have? Look at the smallest and the largest with `value_counts()`.
2. Print 5 random training messages with their intent (`sample(5, random_state=0)`).
3. Print all the intents whose name contains `card` (`train["category"].unique()` and a list comprehension). How easy do you think it is to tell them apart?

In [ ]:
# 1-3. Explore the data

**Question:** what did you notice about the intents with `card` in their name?

*Your answer here.*

## Challenge 1: compare sentences

Start small. Load the model and measure how similar a few pairs of customer messages are.

1. Load `all-MiniLM-L6-v2` into a variable called `model`.
2. For every pair in `pairs`, encode the two sentences and print their cosine similarity, rounded to 2 decimals, next to the pair.

In [ ]:
pairs = [
    ("My card still hasn't arrived", "When will I receive my new card?"),
    ("I was charged twice for the same purchase", "Why is there a duplicate payment on my statement?"),
    ("I was charged twice for the same purchase", "How do I change my PIN?"),
    ("How do I top up my account?", "My top up failed"),
    ("My card was stolen", "My card was not stolen"),
]

# 1. Load the model

# 2. Similarity of each pair

**Checkpoint:** the duplicate-payment pair scores about 0.6 although it shares almost no words, the PIN pair scores about 0.1, and the last pair scores above 0.9.

**Question:** the last pair scores higher than any other. Is that a problem for a support tool? What would happen if a bot answered *My card was not stolen* with the answer to *My card was stolen*?

*Your answer here.*

## Challenge 2: build the index

A search engine encodes all the documents once and keeps the vectors.

1. Encode all the training messages (`train["text"].tolist()`) into `train_vectors`. Use `normalize_embeddings=True` so that every vector has length 1, and `batch_size=64`. It takes between 10 seconds and a minute, depending on your computer.
2. Print the shape of `train_vectors` and the length of the first vector (`np.linalg.norm`).

In [ ]:
# 1. Encode the training messages

# 2. Shape and length

**Checkpoint:** `(10003, 384) 1.0`.

## Challenge 3: semantic search

Complete `semantic_search(query, k=5)`. It must return a DataFrame with the `k` most similar training messages, best first, and three columns: `score`, `category` and `text`.

The steps, as in the lesson:

1. Encode the query with `normalize_embeddings=True`.
2. Compute the similarity with every training message. Because all vectors have length 1, the dot product **is** the cosine similarity: `train_vectors @ query_vector`.
3. Find the positions of the `k` highest scores with `np.argsort` (it sorts from lowest to highest).
4. Build the DataFrame from those positions.

In [ ]:
def semantic_search(query, k=5):
    # 1. Encode the query

    # 2. Similarity with every training message

    # 3. Positions of the k highest scores

    # 4. DataFrame with score, category and text
    pass


semantic_search("I can't find my card anywhere")

**Checkpoint:** the first four results are lost-card messages with the intent `lost_or_stolen_card` and scores above 0.9. The fifth, *Where is my card?*, belongs to `card_arrival`.

## Challenge 4: a TF-IDF baseline

Is the embedding model worth it? Build the Day 1 alternative and compare.

1. Fit a `TfidfVectorizer()` (default settings) on the training texts and keep the training matrix in `train_tfidf`.
2. Write `tfidf_search(query, k=5)`, returning the same three columns as `semantic_search`. TF-IDF rows already have length 1, so the dot product is again the cosine similarity. The product of two sparse matrices is sparse: `.toarray().ravel()` turns the result into a flat NumPy array.
3. Run both searches on each query in `queries`.

In [ ]:
queries = [
    "I can't find my card anywhere",
    "Someone took money from my account without asking",
    "Why did the exchange rate change so much?",
    "The ATM gave me less cash than I asked for",
]

# 1. Fit the vectoriser

# 2. tfidf_search

# 3. Compare

**Checkpoint:** for the first query, TF-IDF's top result is *I can't find my PIN anywhere*.

**Question:** for which queries does TF-IDF go wrong, and why? For which does it do as well as semantic search? Write one query of your own where you expect TF-IDF to fail, and check it.

*Your answer here.*

## Challenge 5: measure it

Four queries are anecdotes. To compare the methods properly, treat the 3,080 test messages as new customer messages, find the most similar training message for each one, and check whether it has the same intent. The share of correct intents is the **top-1 accuracy** of the search.

1. Encode the test messages into `test_vectors` (normalised, as before).
2. `test_vectors @ train_vectors.T` gives a 3,080 × 10,003 matrix with the similarity of every test message to every training message. For each row, `.argmax(axis=1)` gives the position of the most similar training message. Use it to get the predicted intents and compute the accuracy.
3. Do the same with TF-IDF. The similarity matrix is sparse here, and its `argmax(axis=1)` returns a matrix: wrap it with `np.asarray(...).ravel()`.
4. Print both accuracies.

In [ ]:
# 1. Encode the test messages

# 2. Embedding accuracy

# 3. TF-IDF accuracy

# 4. Print both

**Checkpoint:** about 0.92 for embeddings and 0.79 for TF-IDF.

5. Now look at the mistakes of the embedding search. Build a DataFrame with the test text, the true intent and the predicted intent, keep the rows where they differ, and print the 8 most frequent (true, predicted) pairs with `value_counts()`. Then print a few of the mistaken messages.

In [ ]:
# 5. Mistakes of the embedding search

**Question:** look at the most frequent confusions. Are they really mistakes? What does this tell you about the data and about the 0.92?

*Your answer here.*

## Challenge 6: group messages without labels

Imagine the bank had no intents yet, only messages. Could you discover the groups from the embeddings? Test it on six intents, whose true labels you hide from KMeans and only use to check the result.

1. Build a boolean mask of the training messages that belong to the six intents in `six`: `mask = train["category"].isin(six).to_numpy()`. Use it to keep those messages (`train[mask]`) and their vectors (`train_vectors[mask]`).
2. Cluster the vectors with `KMeans(n_clusters=6, n_init=10, random_state=0)`, and show a `pd.crosstab` of the clusters against the true intents.
3. Compute the adjusted Rand index (ARI) of the clustering.
4. Do the same with the TF-IDF rows of those messages (`train_tfidf[mask]`), and compare the two ARIs.

In [ ]:
six = ["card_arrival", "lost_or_stolen_card", "exchange_rate", "top_up_failed", "cash_withdrawal_charge", "passcode_forgotten"]

# 1. Messages and vectors of the six intents

# 2. KMeans and crosstab

# 3. ARI

# 4. The same with TF-IDF

**Checkpoint:** an ARI of about 0.8 for the embeddings and about 0.6 for TF-IDF.

**Question:** which intents does KMeans mix or split, and why might that happen? Would the clusters be useful to a bank that has no intents yet?

*Your answer here.*

## Extra task: know when you don't know

A search tool should not pretend. When even the best match is not very similar, it is better to say *I'm not sure* and pass the message to a person.

1. For each test message, take the score of its best match (`similarity.max(axis=1)`).
2. For the thresholds 0.5, 0.6, 0.7, 0.8 and 0.9, compute (a) the share of test messages whose best score is at least the threshold (the messages the tool would answer) and (b) the top-1 accuracy on those messages only. Present the results as a table.
3. Which threshold would you recommend to the support team, and why?

Bonus: instead of taking the intent of the single best match, take the most common intent among the top 3, 5 and 10 matches (a majority vote, like the k-nearest neighbours classifier). Does it improve the accuracy? Then try a few `TfidfVectorizer` settings from Day 1 (`stop_words="english"`, `ngram_range=(1, 2)`): can any of them close the gap with the embeddings?

In [ ]:
# Extra task

*Your conclusions here.*